# Jet Classifier — Interactive Notebook (exploratory)

Runs the same jet-tagging exercise as [Hands-On Prep](../../lessons/3_prep.md) / [Hands-On Exercise](../../lessons/4_hands_on.md) — training and evaluating a classifier on the `hls4ml_lhc_jets_hlf` dataset — entirely inline in this notebook instead of as a Kubernetes Job. Adapted from [Javier Duarte's PHYS 139/239 course at UCSD](https://jduarte.physics.ucsd.edu/phys139_239/03_Tabular_Data_NN.html), the same source material the Job-based version credits.

**This is exploratory, not a numbered lesson yet.** It's here to show that NRP's JupyterHub itself can be GPU-backed — you don't need to submit a Kubernetes Job just to train something interactively. The Job-based path (lessons 3-4) earns its keep once you want reproducibility, want to close your laptop while it runs, or want to run many configurations in parallel like the [hyperparameter sweep extension](../../lessons/4_hands_on.md#extension-hyperparameter-sweep-optional) — a single notebook session can't do that concurrency by itself.

A few things are different from the rest of this training:

- **Python kernel**, like `5_cms_data.ipynb` and `dask_hep_example.ipynb` — not the bash kernel used elsewhere.
- **No Kubernetes at all.** No PVC, no Job YAML, no `kubectl`. Everything happens in this process, and results are saved to your home directory on the hub, not the shared training PVC.
- **Needs a GPU-backed server to be fast.** The model below is intentionally oversized for a GPU — same reasoning as [Hands-On Prep](../../lessons/3_prep.md#what-youre-building-a-jet-classifier): no scientific motivation, purely to turn a CPU-fast toy into something that exercises a GPU. On a CPU-only server this still runs, just much slower.

## Install dependencies

Unlike the Job-based version (which uses a custom prebuilt image with TensorFlow already installed), this notebook's environment isn't guaranteed to have the ML stack. Check first and install into your user site-packages if it's missing:

In [ ]:
import importlib.util
import os
import subprocess
import sys

# cwd=~ avoids a pip bug where it crashes if the process's working directory
# no longer exists (os.getcwd() -> FileNotFoundError), which can happen on some
# JupyterHub setups.
home = os.path.expanduser("~")
packages = [
    ("tensorflow", "tensorflow"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("numpy", "numpy"),
]
for import_name, pip_name in packages:
    if importlib.util.find_spec(import_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--user", "--quiet", pip_name],
            check=True, cwd=home,
        )

# Some hub images (notably conda-based ones) don't put the --user
# site-packages directory on sys.path by default, so a successful pip
# install can still leave the import failing. Make sure it's there.
import site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
print("Dependencies ready")


## Make sure TensorFlow can find the GPU driver

On some GPU-backed servers, the NVIDIA driver's libraries get mounted into the container *after* the image's library cache was already built, so `libcuda.so.1` is physically present but not on the path anything doing a plain `dlopen()` will check. `nvidia-smi` never hits this — it talks to the driver directly through NVML, not `dlopen()` — so it can report your GPU just fine while TensorFlow still logs `Could not find cuda drivers on your machine` and silently falls back to CPU. This cell looks for `libcuda.so*` in the usual places and adds its directory to `LD_LIBRARY_PATH` if it's missing, before TensorFlow ever gets imported.

In [ ]:
import ctypes.util
import glob
import os

def _ensure_libcuda_on_path():
    if ctypes.util.find_library("cuda"):
        return  # already discoverable, nothing to do
    candidates = sorted(
        glob.glob("/usr/lib/x86_64-linux-gnu/libcuda.so*")
        + glob.glob("/usr/lib*/libcuda.so*")
        + glob.glob("/usr/local/nvidia/lib*/libcuda.so*")
        + glob.glob("/usr/local/cuda*/lib64/libcuda.so*")
    )
    if not candidates:
        print("No libcuda.so found in the usual locations — if you have a GPU, TensorFlow may "
              "still fail to find it; see the notebook markdown above for how to look further.")
        return
    directory = os.path.dirname(candidates[0])
    current = os.environ.get("LD_LIBRARY_PATH", "")
    if directory not in current.split(":"):
        os.environ["LD_LIBRARY_PATH"] = f"{directory}:{current}" if current else directory
        print(f"Added {directory} to LD_LIBRARY_PATH (found {candidates[0]})")

_ensure_libcuda_on_path()


## Check for a GPU

If this prints `none`, everything below still works — it'll just be a lot slower. On the Analysis Hub, make sure you picked a GPU-backed server profile when you started your session.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs visible to TensorFlow: {[g.name for g in gpus] or 'none'}")


## Hyperparameters

Same defaults as the Job-based version's `MODEL_WIDTHS`/`BATCH_SIZE`/`MIXED_PRECISION`, so results are comparable — but as plain variables here instead of environment variables, so feel free to edit and re-run. `EPOCHS` is lower than the Job's default of 50, since you're watching this run live instead of it running in the background — bump it up if you want closer-to-final accuracy. Javier's original teaching example used a much smaller `(64, 32, 32)` — try that too if you want to see the "no scientific motivation" claim for yourself: similar accuracy, far less GPU work.

In [ ]:
SEED = 42
EPOCHS = 20
BATCH_SIZE = 8192
LEARNING_RATE = 0.001
MODEL_WIDTHS = (4096, 4096, 2048, 1024)   # try (64, 32, 32) for Javier's original teaching-sized model
MIXED_PRECISION = True
TEST_FRACTION = 0.2
VALIDATION_FRACTION = 0.25

RESULTS_DIR = os.path.expanduser("~/jet-class-notebook-results")
os.makedirs(RESULTS_DIR, exist_ok=True)


## Load the dataset

[`hls4ml_lhc_jets_hlf`](https://www.openml.org/search?type=data&id=42468): roughly 830,000 simulated jets, 16 high-level features, 5 classes (gluon, light quark, W, Z, top quark) — see [Hands-On Prep](../../lessons/3_prep.md#what-youre-building-a-jet-classifier) for the physics background. Always the full dataset — no sampling.

In [ ]:
import random
import shutil
import time

import numpy as np
from sklearn.datasets import fetch_openml, get_data_home

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# The OpenML download occasionally arrives corrupted somewhere on the network
# path out of the cluster (a different wrong checksum each time, not a
# stale/bad file upstream) — scikit-learn's own single built-in retry isn't
# always enough. Wipe the cache and try again a few times before giving up.
data = None
last_error = None
for attempt in range(1, 6):
    try:
        data = fetch_openml("hls4ml_lhc_jets_hlf", parser="auto", cache=True)
        break
    except ValueError as exc:
        last_error = exc
        print(f"OpenML download attempt {attempt}/5 failed (likely transit "
              f"corruption, not a bad upstream file): {exc}")
        shutil.rmtree(get_data_home(), ignore_errors=True)
        if attempt < 5:
            time.sleep(5 * attempt)
if data is None:
    raise RuntimeError("Failed to download hls4ml_lhc_jets_hlf from OpenML "
                        "after 5 attempts") from last_error

X_df = data["data"]
y = data["target"]
feature_names = list(data["feature_names"])

print(f"Feature names: {feature_names}")
print(f"Shapes: X={X_df.shape}, y={y.shape}")
X_df.head()


## Preprocess

Encode labels, split into train/validation/test, and standardize the features:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

X_np = X_df.to_numpy(dtype=np.float32, copy=True)
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y).astype(np.int16)
y_onehot = to_categorical(y_encoded, num_classes=len(encoder.classes_)).astype(np.float32)
classes = encoder.classes_.tolist()
print(f"Classes: {classes}")

X_train_val, X_test, y_train_val, y_test, labels_train_val, labels_test = train_test_split(
    X_np, y_onehot, y_encoded,
    test_size=TEST_FRACTION, random_state=SEED, stratify=y_encoded,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=VALIDATION_FRACTION, random_state=SEED, stratify=labels_train_val,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print(f"X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")

def make_dataset(features, labels, *, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((features, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(features), 100_000), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    else:
        ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(X_train, y_train, shuffle=True)
val_ds = make_dataset(X_val, y_val, shuffle=False)
test_ds = make_dataset(X_test, y_test, shuffle=False)


## Build the model

A plain fully-connected network — one `Dense` layer per width in `MODEL_WIDTHS`, softmax output. Mixed precision (`float16` compute, `float32` output) is what actually makes this GPU-heavy rather than just wide. Mixed precision only gets applied if a GPU was actually detected above — it's a GPU-specific optimization that tends to make CPU training slower, not faster, so it's skipped automatically on a CPU-only server.

In [ ]:
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

if MIXED_PRECISION and gpus:
    mixed_precision.set_global_policy("mixed_float16")
elif MIXED_PRECISION:
    print("No GPU detected — skipping mixed_float16 (a GPU optimization; on CPU it typically adds overhead instead of helping).")

model = Sequential(name="jet_classifier")
model.add(Input(shape=(X_train.shape[1],), name="features"))
for idx, width in enumerate(MODEL_WIDTHS, start=1):
    model.add(Dense(width, activation="relu", name=f"dense_{idx}"))
model.add(Dense(y_onehot.shape[1], activation="softmax", dtype="float32", name="class_probabilities"))
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## Train

Watch the loss/accuracy print live, epoch by epoch — the thing you can't do with a backgrounded Job:

In [ ]:
import time

train_start = time.time()
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    verbose=2,
)
training_seconds = time.time() - train_start
print(f"Training took {training_seconds:.1f}s ({model.count_params():,} parameters)")


## Evaluate

Predict on the held-out test set and check accuracy:

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(test_ds, verbose=0)
accuracy = accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_pred, axis=1))
print(f"Test accuracy: {accuracy:.4f}")


### Training history

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
fig.tight_layout()
plt.show()


### Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

y_true_idx = np.argmax(y_test, axis=1)
y_pred_idx = np.argmax(y_pred, axis=1)
matrix = confusion_matrix(y_true_idx, y_pred_idx, normalize="true")

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(matrix, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
ax.set(
    xticks=np.arange(len(classes)), yticks=np.arange(len(classes)),
    xticklabels=classes, yticklabels=classes,
    ylabel="True label", xlabel="Predicted label", title="Normalized confusion matrix",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
for row in range(matrix.shape[0]):
    for col in range(matrix.shape[1]):
        color = "white" if matrix[row, col] > 0.5 else "black"
        ax.text(col, row, f"{matrix[row, col]:.2f}", ha="center", va="center", color=color)
fig.tight_layout()
plt.show()


### ROC curves

In [ ]:
from sklearn.metrics import auc, roc_curve

fig, ax = plt.subplots(figsize=(7, 6))
for idx, label in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test[:, idx], y_pred[:, idx])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=1.8, label=f"{label} (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curves")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()


## Save the model

Saved locally to your home directory on the hub — no PVC, no `kubectl cp` needed, since nothing here ever left this notebook's process:

In [ ]:
model_path = os.path.join(RESULTS_DIR, "jet_classifier.keras")
model.save(model_path)
print(f"Saved model to {model_path}")


## Where to go from here

You just ran the same exercise as [Hands-On Exercise](../../lessons/4_hands_on.md), interactively, with no Kubernetes involved at all. That's genuinely useful for quick iteration — tweak `MODEL_WIDTHS` or `EPOCHS` above and re-run a couple cells to see what changes.

Where a notebook session stops being the right tool: you want the result to keep running after you close your laptop, you want a clean reproducible artifact independent of *this* notebook's state, or you want to compare several configurations *at once* rather than one after another. That's exactly what the [hyperparameter sweep extension](../../lessons/4_hands_on.md#extension-hyperparameter-sweep-optional) does with four parallel Kubernetes Jobs — same underlying training code, just handed to the cluster instead of run here.